# Cleaning for Order Items Data:

---

In [0]:
df = spark.table("project.bronze.bronze_order_items")
display(df.head(4))

# Checking the schema for data type mismatch
df.printSchema()

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,ingestion_timestamp
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.9,13.29,2026-03-09T13:55:34.780Z
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.9,19.93,2026-03-09T13:55:34.780Z
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.0,17.87,2026-03-09T13:55:34.780Z
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18.000Z,12.99,12.79,2026-03-09T13:55:34.780Z


root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



In [0]:
# calculating the null values
{col: df.filter(df[col].isNull()).count() for col in df.columns}

{'order_id': 0,
 'order_item_id': 0,
 'product_id': 0,
 'seller_id': 0,
 'shipping_limit_date': 0,
 'price': 0,
 'freight_value': 0,
 'ingestion_timestamp': 0}

### Observation: There are no null values

Now, we have to:
- Drop the duplicates for `order_id` and `order_item_id`
- validate the `price` and `freight_value` cols
- ensures schema consistency for timestamp cols

In [0]:
from pyspark.sql.functions import col, to_timestamp

df_order_items_silver = (
    df
    .dropDuplicates(["order_id", "order_item_id"])
    .filter((col("price") >= 0) & (col("freight_value") >= 0))
    .withColumn("shipping_limit_date", to_timestamp(col("shipping_limit_date")))
    .drop("order_item_id","seller_id")
)


In [0]:
# saving the cleaned data
df_order_items_silver.write.format('delta').mode('overwrite').saveAsTable("project.silver.silver_items")


# Now, Cleaning for Order Payments Data:
---

In [0]:
# loading the data and displaying top 5 rows
df_payments = spark.table("dai.project.bronze_order_payments")
display(df_payments.limit(5))

order_id,payment_sequential,payment_type,payment_installments,payment_value,ingestion_timestamp
b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33,2026-03-08T09:53:36.789Z
a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39,2026-03-08T09:53:36.789Z
25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71,2026-03-08T09:53:36.789Z
ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78,2026-03-08T09:53:36.789Z
42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45,2026-03-08T09:53:36.789Z


In [0]:
# print schema to check any mismatch of datatype, for example, timestamp being string.
df_payments.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



In [0]:
# calculating the null values
{col: df_payments.filter(df_payments[col].isNull()).count() for col in df_payments.columns}

{'order_id': 0,
 'payment_sequential': 0,
 'payment_type': 0,
 'payment_installments': 0,
 'payment_value': 0,
 'ingestion_timestamp': 0}

### No Null Values Present

Performing cleaning operation as done above

In [0]:
from pyspark.sql.functions import col, lower, trim

df_payments_silver = (
    df_payments
    .dropDuplicates(["order_id", "payment_sequential"])
    .filter((col("payment_value") > 0) & (col("payment_installments") >= 1))
    .withColumn("payment_type", lower(trim(col("payment_type"))))
    .drop("payment_sequential")
)

In [0]:
# saving the cleaned data
df_payments_silver.write.format('delta').mode('overwrite').saveAsTable("project.silver.silver_payments")